In [ ]:
# Cell 1 — Mount Drive and install packages
from google.colab import drive
drive.mount('/content/drive')
DRIVE_BASE = '/content/drive/MyDrive/PhishGuard'

import os, subprocess
subprocess.run(['pip', 'install',
    'xgboost==2.1.0', 'scikit-learn==1.5.0',
    'pandas==2.2.0', 'numpy==1.26.0', 'joblib==1.4.0', '-q'], check=False)

assert os.path.exists(f'{DRIVE_BASE}/features/wallet_features.csv'), \
    'wallet_features.csv not found — run Notebook 02 first'
assert os.path.exists(f'{DRIVE_BASE}/models/wallet_feature_schema.json'), \
    'wallet_feature_schema.json not found — run Notebook 02 first'
print('Cell 1 ready.')


In [ ]:
# Cell 2 — Imports and constants
import pandas as pd
import numpy as np
import json
import joblib
import xgboost as xgb
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import (classification_report, confusion_matrix,
                              roc_auc_score, average_precision_score)

DATA_PATH   = f'{DRIVE_BASE}/features/wallet_features.csv'
SCHEMA_PATH = f'{DRIVE_BASE}/models/wallet_feature_schema.json'
MODEL_PATH  = f'{DRIVE_BASE}/models/wallet_xgboost_v1.pkl'
EVAL_DIR    = f'{DRIVE_BASE}/evaluation'
os.makedirs(EVAL_DIR, exist_ok=True)

RANDOM_SEED = 42
TEST_SIZE   = 0.20

print('Cell 2 imports loaded.')


In [ ]:
# Cell 3 — Load data and validate
df = pd.read_csv(DATA_PATH)
with open(SCHEMA_PATH) as f:
    FEATURE_COLS = json.load(f)

assert df.shape == (4000, 25),                         f'Unexpected shape: {df.shape}'
assert len(FEATURE_COLS) == 23,                        f'Schema has {len(FEATURE_COLS)} features, expected 23'
assert df['label'].value_counts()[1] == 2000,          'Expected 2000 phishing rows'
assert df['label'].value_counts()[0] == 2000,          'Expected 2000 benign rows'
assert df.isnull().sum().sum() == 0,                   'Nulls found in features'
assert not np.isinf(df[FEATURE_COLS].values).any(),    'Inf values found in features'

X = df[FEATURE_COLS].values.astype(np.float32)
y = df['label'].values.astype(np.int32)

print(f'Loaded: {df.shape}')
print(f'Features: {len(FEATURE_COLS)}')
print(f'Class distribution: {dict(zip(*np.unique(y, return_counts=True)))}')


In [ ]:
# Cell 4 — Train / test split
all_idx_w = np.arange(len(y))
train_idx_w, test_idx_w = train_test_split(
    all_idx_w, test_size=TEST_SIZE, random_state=RANDOM_SEED, stratify=y
)
X_train, X_test = X[train_idx_w], X[test_idx_w]
y_train, y_test = y[train_idx_w], y[test_idx_w]

np.save(f'{DRIVE_BASE}/models/wallet_test_indices.npy', test_idx_w)
print(f'Saved wallet_test_indices.npy — {len(test_idx_w)} rows')


print(f'Train: {X_train.shape} — phishing: {y_train.sum():,} | benign: {(y_train==0).sum():,}')
print(f'Test:  {X_test.shape}  — phishing: {y_test.sum():,}  | benign: {(y_test==0).sum():,}')


In [ ]:
# Cell 5 — Train XGBoost model
params = dict(
    n_estimators     = 400,
    max_depth        = 6,
    learning_rate    = 0.05,
    subsample        = 0.8,
    colsample_bytree = 0.8,
    min_child_weight = 3,
    gamma            = 0.1,
    reg_alpha        = 0.1,
    reg_lambda       = 1.0,
    scale_pos_weight = 1,     # dataset is balanced — no adjustment needed
    eval_metric      = 'auc',
    random_state     = RANDOM_SEED,
    n_jobs           = -1,
)

model = xgb.XGBClassifier(**params)
model.fit(
    X_train, y_train,
    eval_set=[(X_train, y_train), (X_test, y_test)],
    verbose=50
)
print('Training complete.')


In [ ]:
# Cell 6 — Evaluate: metrics, cross-validation, SHAP
y_pred      = model.predict(X_test)
y_pred_prob = model.predict_proba(X_test)[:, 1]

print('=== CLASSIFICATION REPORT ===')
print(classification_report(y_test, y_pred, target_names=['Benign', 'Phishing'], digits=4))

cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()
print(f'Confusion matrix — TN: {tn}  FP: {fp}  FN: {fn}  TP: {tp}')

roc_auc  = roc_auc_score(y_test, y_pred_prob)
avg_prec = average_precision_score(y_test, y_pred_prob)
print(f'ROC-AUC:       {roc_auc:.4f}')
print(f'Avg Precision: {avg_prec:.4f}')

# 5-fold stratified CV on full dataset
cv        = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
cv_scores = cross_val_score(
    xgb.XGBClassifier(**params), X, y,
    cv=cv, scoring='roc_auc', n_jobs=-1)
print(f'5-fold CV ROC-AUC: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')
print(f'Per-fold:          {[round(s, 4) for s in cv_scores]}')


In [ ]:
# Cell 7 — Save model and verify
joblib.dump(model, MODEL_PATH)
print(f'Saved: {MODEL_PATH}')

# Reload and verify predictions are identical
model_check  = joblib.load(MODEL_PATH)
y_check      = model_check.predict(X_test)

assert np.array_equal(y_check, y_pred), \
    'Reloaded model predictions differ from original'
assert model_check.n_features_in_ == len(FEATURE_COLS), \
    f'Feature count mismatch: {model_check.n_features_in_} vs {len(FEATURE_COLS)}'

print(f'Verified:             {MODEL_PATH}')
print(f'Model feature count:  {model_check.n_features_in_}')
print(f'XGBoost version:      {xgb.__version__}')
print('Notebook 04 complete.')
